# Episodic Memory — Selective Recall with Similarity Search

This notebook demonstrates `SimilarityMemory` and `QdrantMemoryStore` working together
to give an `LLMAgent` episodic memory that retrieves past episodes by **semantic
relevance** rather than recency.

Two properties are illustrated:

- **Selective recall.** After each task the agent records an episode in a Qdrant
  in-memory vector store. When a new task arrives, `SimilarityMemory` searches for
  the most semantically similar past episodes — not the most recent. A question about
  Southeast Asia retrieves Southeast Asian country episodes; a question about European
  economies retrieves European ones, regardless of when they were recorded.
- **Contrast with recency.** The notebook explicitly shows what `RecencyMemory` would
  have returned for the same queries — the same three most-recent episodes for both —
  illustrating exactly when similarity search adds value over a simpler recency strategy.

> **This notebook uses the [REST Countries API](https://restcountries.com/)** — free,
> no authentication required.

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama installed
on your local machine and have its LLM hosting service running. To download Ollama,
follow the instructions found on this page: https://ollama.com/download. After
downloading and installing Ollama, you can start a service by opening a terminal
and running the command `ollama serve`.

## Setup

In [ ]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"✓ Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"✓ Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


ensure_ollama()

In [ ]:
import json
import logging
import urllib.error
import urllib.parse
import urllib.request

from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.logger import enable_console_logging
from llm_agents_from_scratch.memory import QdrantMemoryStore, SimilarityMemory
from llm_agents_from_scratch.tools.simple_function import SimpleFunctionTool

enable_console_logging(logging.INFO)

## Defining the Tool

The [REST Countries API](https://restcountries.com/) returns geographic and
demographic facts for any country by name — no API key required.

In [ ]:
def get_country(name: str) -> str:
    """Look up a country by name and return key facts.

    Uses the REST Countries API (https://restcountries.com/). No auth required.
    """
    encoded = urllib.parse.quote(name.strip())
    url = f"https://restcountries.com/v3.1/name/{encoded}?fullText=true"
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "llm-agents-from-scratch/1.0"},
    )
    try:
        with urllib.request.urlopen(req) as resp:
            data = json.loads(resp.read())[0]
    except urllib.error.HTTPError as e:
        if e.code == 404:  # noqa: PLR2004
            raise ValueError(
                f"Country '{name}' not found. "
                "Check the spelling and try again.",
            ) from e
        raise
    return json.dumps(
        {
            "name": data["name"]["common"],
            "region": data["region"],
            "subregion": data.get("subregion", ""),
            "capital": data.get("capital", [""])[0],
            "population": data["population"],
            "area_km2": data.get("area", 0),
            "languages": list(data.get("languages", {}).values()),
            "currencies": [
                v["name"] for v in data.get("currencies", {}).values()
            ],
        },
    )


get_country_tool = SimpleFunctionTool(func=get_country)

## Creating the Agent with Memory

`QdrantMemoryStore` runs Qdrant in-process — no server required. Episodes are
embedded at write time using FastEmbed (local ONNX, no external service needed).
`SimilarityMemory` wraps the store and decides what to recall (the `k` most
semantically similar episodes) and how to format the result for the system prompt.
The agent holds the memory in its `memories` list. `TaskHandler` calls `recall`
at task start and `record` at task end automatically.

In [ ]:
store = QdrantMemoryStore()
memory = SimilarityMemory(store=store, k=3)
llm = OllamaLLM(model="qwen3:14b", think=False)
agent = LLMAgent(llm=llm, tools=[get_country_tool], memories=[memory])

## Part 1 — Building Up Episodic Memory

Seven country lookups spanning four distinct regions: Southeast Asia (Thailand,
Vietnam, Indonesia), Western Europe (France, Germany), East Africa (Kenya), and
South America (Brazil). Each task calls `get_country` once and records an episode
at the end. After all lookups, `memory.summary()` shows the store state including
the newest and oldest episodes.

In [ ]:
task1 = Task(
    instruction=(
        "What are Thailand's key geographic and demographic facts? "
        "Use the get_country tool. Do not rely on prior knowledge."
    ),
)
result1 = await agent.run(task1)
print(result1)

In [ ]:
task2 = Task(
    instruction=(
        "What are Vietnam's key geographic and demographic facts? "
        "Use the get_country tool. Do not rely on prior knowledge."
    ),
)
result2 = await agent.run(task2)
print(result2)

In [ ]:
task3 = Task(
    instruction=(
        "What are Indonesia's key geographic and demographic facts? "
        "Use the get_country tool. Do not rely on prior knowledge."
    ),
)
result3 = await agent.run(task3)
print(result3)

In [ ]:
print(await memory.summary())

In [ ]:
task4 = Task(
    instruction=(
        "What are France's key geographic and demographic facts? "
        "Use the get_country tool. Do not rely on prior knowledge."
    ),
)
result4 = await agent.run(task4)
print(result4)

In [ ]:
task5 = Task(
    instruction=(
        "What are Germany's key geographic and demographic facts? "
        "Use the get_country tool. Do not rely on prior knowledge."
    ),
)
result5 = await agent.run(task5)
print(result5)

In [ ]:
task6 = Task(
    instruction=(
        "What are Kenya's key geographic and demographic facts? "
        "Use the get_country tool. Do not rely on prior knowledge."
    ),
)
result6 = await agent.run(task6)
print(result6)

In [ ]:
task7 = Task(
    instruction=(
        "What are Brazil's key geographic and demographic facts? "
        "Use the get_country tool. Do not rely on prior knowledge."
    ),
)
result7 = await agent.run(task7)
print(result7)

In [ ]:
print(await memory.summary())

## The Recency Baseline

Before running any synthesis tasks, it is worth checking what `RecencyMemory` would
recall for any query at this point — regardless of topic. The three most recently
recorded episodes are Kenya and Brazil (the last two lookups), plus whatever came
just before. A recency strategy would inject these same three episodes into the
system prompt for *both* the Southeast Asia query and the European economy query
that follow.

In [ ]:
print("Most recent 3 episodes (RecencyMemory baseline for any query):\n")
most_recent = await memory.store.read_recent(3)
for ep in most_recent:
    print(f"  - {ep.task.instruction[:65]}")

## Part 2 — Selective Recall: Southeast Asia

The synthesis task asks about Southeast Asian countries. No country facts are
provided in the instruction — the agent can only answer from recalled episodes
injected into its system prompt. `SimilarityMemory` searches the store for the
three episodes most semantically similar to the query and injects only those.
Watch the logs: you should see no `🛠️ Executing Tool Call` lines.

In [ ]:
task8 = Task(
    instruction=(
        "Which of the countries you have researched are in Southeast Asia? "
        "Summarise their key facts and compare their populations. "
        "You do not need to call any tools. Use only what you already know."
    ),
)
result8 = await agent.run(task8)
print(result8)

The cell below shows exactly which episodes were retrieved for this query. Compare
this to the recency baseline above — similarity search surfaces Thailand, Vietnam,
and Indonesia rather than the three most recently recorded countries.

In [ ]:
print("Episodes recalled for the Southeast Asia query:\n")
recalled_sea = await memory.store.search(
    query=task8.instruction,
    k=memory.k,
)
for ep in recalled_sea:
    print(str(ep))
    print()

## Part 3 — A Different Query, A Different Subset: European Economies

A second synthesis task asks about European economies. The query is semantically
distant from Southeast Asia, so a completely different subset of episodes should
be recalled — France and Germany — even though they were recorded earlier than
Kenya and Brazil.

In [ ]:
task9 = Task(
    instruction=(
        "Which of the countries you have researched are in Western Europe? "
        "How do they compare in population and area? "
        "You do not need to call any tools. Use only what you already know."
    ),
)
result9 = await agent.run(task9)
print(result9)

In [ ]:
print("Episodes recalled for the European economies query:\n")
recalled_eu = await memory.store.search(
    query=task9.instruction,
    k=memory.k,
)
for ep in recalled_eu:
    print(str(ep))
    print()

## Key Takeaway

`SimilarityMemory` and `QdrantMemoryStore` give the agent something recency-based
memory cannot: **context that is relevant to the current task**, not merely recent.

The two queries issued in Parts 2 and 3 touched completely different regions. A
recency strategy would have injected Kenya and Brazil into both — neither useful for
Southeast Asia, nor for Western Europe. Similarity search retrieved the right subset
for each query, even though those episodes were recorded earlier in the session.

The separation of concerns is the same as in `RecencyMemory`:
`SimilarityMemory` decides *what* to retrieve and *how* to format it for the prompt;
`QdrantMemoryStore` decides *how* to persist and index episodes. Swapping one does
not affect the other: the same `SimilarityMemory` works over any `BaseMemoryStore`
that implements `search()`, and the same `QdrantMemoryStore` works with any
`BaseMemory` implementation.